<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union
import re
from datetime import datetime

## SQLAlchemy imports
from sqlalchemy import create_engine, MetaData, Table, Column, DateTime as sqlalchemy_DateTime, Index, insert, text as sqlalchemy_text, func as sqlalchemy_func, select as sqlalchemy_select
from sqlalchemy import types as generic_sql_types

# aux packages
from IPython.display import Image
import logging
import pathlib
import importlib.util


# set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# folder using CWD as base
curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

image_folder = curr_working_dir / 'src' / 'images'
if not image_folder.exists():
    image_folder.mkdir(parents=True)
    logger.info(f"Created image folder at: {image_folder}")


# Obtain resource files

In [ ]:


#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 3 {file_name}

In [ ]:
# extract, parse text files into pandas df. ignore first row with date (i.e. 31DEC2007)
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
barra_df_08 = pd.read_csv('USE3L0812.RSK', skiprows=1, header=0)

In [ ]:
# check shape of each dataframe
print(f"{barra_df_07.shape=}")
print(f"{barra_df_08.shape=}")
# assert columns match
assert barra_df_07.columns.equals(barra_df_08.columns), "Columns of the two dataframes do not match."
pd.testing.assert_index_equal(barra_df_07.columns, barra_df_08.columns, check_names=True) # secondary way using pd testing module

In [ ]:
display(barra_df_07.head(3))
display(barra_df_08.head(3))

In [ ]:
# look for null counts
for df in [barra_df_07, barra_df_08]:
    print(f"{df.isnull().values.any()=}")

## Cleaning Dataframe:
- remove white spaces from:
    - column names
    - string values in columns
    - index values

In [ ]:
def clean_index_whitespace(df):
    match df.index:
        case pd.Index() as idx if idx.dtype == 'O': # object dtype, likely string index
            df.index = df.index.str.strip()
            logger.info(f"stripped whitespace from {df.index.name=}")
        case _:
            logging.info(f"Index {idx.dtype=}. Skipping")
    return df

def clean_col_whitespace(df):
    match df.columns.dtype:
        case dtype if dtype == 'O':
            # create the stripped version
            original_columns = df.columns
            stripped_columns = original_columns.str.strip()

            # find the difference to output
            changed_cols_mask = original_columns != stripped_columns
            cleaned_cols = original_columns[changed_cols_mask].tolist()

            if cleaned_cols:
                logging.info(f"Stripped whitespace from columns: {cleaned_cols}")
            else:
                logging.info("No columns had leading/trailing whitespace to clean.")

            df.columns = stripped_columns
        case _:
            logging.info(f"skipped: {df.columns.dtype=} is not object dtype")
    return df


In [ ]:
# clean df index for both dataframes
barra_df_07 = clean_index_whitespace(barra_df_07)
barra_df_08 = clean_index_whitespace(barra_df_08)

# clean df columns for both dataframes
barra_df_07 = clean_col_whitespace(barra_df_07)
barra_df_08 = clean_col_whitespace(barra_df_08)

In [ ]:
# given a dataframe, and dict of invalid chars, replace invalid chars
def replace_invalid_chars_in_cols(df, invalid_char_dict): 
    for invalid_char, replacement in invalid_char_dict.items():
        df.columns = df.columns.str.replace(invalid_char, replacement, regex=False)
    return df

In [ ]:
def make_col_lower_case(df):
    df.columns = df.columns.str.lower()
    return df

In [ ]:
# apply replace_invalid_chars and make_col_lower_case to both dataframes
invalid_characters = {"%": "_pct"}
barra_df_07 = replace_invalid_chars_in_cols(barra_df_07, invalid_characters)
barra_df_08 = replace_invalid_chars_in_cols(barra_df_08, invalid_characters)

barra_df_07 = make_col_lower_case(barra_df_07)
barra_df_08 = make_col_lower_case(barra_df_08)

In [ ]:
# check df heads after cleaning
display(barra_df_07.columns)
display(barra_df_08.columns)

In [ ]:
# review data to see if it has been scaled, considering whether its a percentage
def review_numeric_cols_for_data_scaling(df, thresholds, exclude_pattern=None, precision=4):
    """
    looks at numeric columns in a dataframe.
    checks it against a provided threshold, client to provide. 
    Example: 
    thresholds = {'percentage': 150, 'std_dev': 5}
        if the column is a percentage, then a reasonable range would be -150 to 150.
        otherwise, we should expect to see between -5 and +5 standard deviations. 
    """
    findings = list()
    pd.options.display.float_format = f'{{:.{precision}f}}'.format # set float display precision for output

    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        col_min = round(df[col].min(), precision)
        col_max = round(df[col].max(), precision)

        is_pct_match = bool(exclude_pattern and re.search(exclude_pattern, col, re.IGNORECASE))

        match is_pct_match:
            case True:
                limit = thresholds.get('percentage', 150) # default to 150 if not specified
            case False:
                limit = thresholds.get('std_dev', 5) # default to 5 if not specified

        if col_max > limit or col_min < -limit:
            findings.append({
                'column': col,
                'min': col_min,
                'max': col_max,
                'limit_used': limit,
                'is_pct': is_pct_match,
                'precision': precision
            })
    return pd.DataFrame(findings)

In [ ]:
SCALING_THRESHOLDS = {'percentage': 200, 'std_dev': 10}

In [ ]:
# review numeric columns for data scaling, exclude percentage to review separately
num_findings_07 = review_numeric_cols_for_data_scaling(barra_df_07, SCALING_THRESHOLDS, exclude_pattern='_pct', precision=4)

print(f"Num columns with potential scaling issues (07):\n {num_findings_07[num_findings_07['is_pct'] == False]}")
print(f"Pct columns with potential scaling issues (07):\n {num_findings_07[num_findings_07['is_pct'] == True]}")



In [ ]:
num_findings_08 = review_numeric_cols_for_data_scaling(barra_df_08, SCALING_THRESHOLDS, exclude_pattern='_pct', precision=4)
print(f"Num columns with potential scaling issues (08):\n {num_findings_08[num_findings_08['is_pct'] == False]}")
print(f"Pct columns with potential scaling issues (08):\n {num_findings_08[num_findings_08['is_pct'] == True]}")

Findings: <br>
- What we noticed was that ind1-5 do not appear to be meaningful numeric values. These are the sector mappings. 
- Price and Capitalization are in line with expectations. We were looking to see if there was negative values or setinel values to represent NULL.
- hbta does have a setinel value. -999 needs to be replaced with NULL.
- yld_pct in 07 is interesting as it has %50,450 max value that will be further investigated.

In [ ]:
def faceted_box_plots(
        df,
        y_column,
        facet_col,
        facet_col_wrap=4,
        base_row_height=300,
        template = 'plotly_dark',
        points = 'outliers',
        facet_col_spacing = 0.02,
        facet_row_spacing = 0.02,
        title = None
):
    # calc number of rows neeeded for facets 
    num_facets = df[facet_col].nunique()

    # avoid division by zero if only one facet
    facet_col_wrap = max(1, facet_col_wrap)
    num_rows = math.ceil(num_facets / facet_col_wrap)

    # adjust height
    total_height = base_row_height * num_rows

    fig = px.box(
        df,
        y=y_column,
        facet_col=facet_col,
        facet_col_wrap=facet_col_wrap,
        height=total_height,
        points=points,
        template=template,
        title=title,
        facet_col_spacing=facet_col_spacing,
        facet_row_spacing=facet_row_spacing
    )

    fig.update_yaxes(matches=None, showticklabels=True) # ensure each facet has its own y-axis range and shows tick labels

    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) # simplify facet labels to just the category value

    return fig


In [ ]:
col_to_include_07 = ['hbta', 'yld_pct']
df_07_filtered = barra_df_07[col_to_include_07]
df_07_filtered_melted = barra_df_07.melt(
    value_vars=col_to_include_07,
    var_name='factor_type',
    value_name='factor_value'
)
fig_07 = faceted_box_plots(
    df_07_filtered_melted,
    y_column='factor_value',
    facet_col='factor_type',
    title='Distribution of Numeric Factors with Potential Scaling Issues (07 Data)'
)
fig_07.show()

In [ ]:
# find the stock with yld_pct > 50K to investigate
outlier_yld_pct_07 = barra_df_07[barra_df_07['yld_pct'] > 50000]
outlier_yld_pct_07[['ticker', 'yld_pct', 'e3estu', 'price', 'capitalization']]

In our analysis, NEWCQ will be filtered out because it is not part of Barra's Estimation Universe. This is a penny stock so the outlier percentage makes sense.

In [ ]:
# 08 data
cols_to_include = ['hbta']
df_08_filtered = barra_df_08[cols_to_include]
df_08_filtered_melted = barra_df_08.melt(
    value_vars=cols_to_include,
    var_name='factor_type',
    value_name='factor_value'
)
fig_08 = faceted_box_plots(
    df_08_filtered_melted,
    y_column='factor_value',
    facet_col='factor_type',
    title='Distribution of Numeric Factors with Potential Scaling Issues (08 Data)'
)
fig_08.show()

In [ ]:
# get null counts for hbta column in both dataframes
hbta_null_count_07 = (barra_df_07['hbta'] == -999).sum()
hbta_null_count_08 = (barra_df_08['hbta'] == -999).sum()

print(f"Null count for hbta in 07 data: {hbta_null_count_07}")
print(f"Null count for hbta in 08 data: {hbta_null_count_08}")

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)
barra_df_08['hbta'] = barra_df_08['hbta'].replace(-999, np.nan)

In [ ]:
# plot 07 hbta distribution after replacing -999 with np.nan
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN (07 Data)'
)
fig.show()

In [ ]:
# plot 08 hbta distribution after replacing -999 with np.nan
fig = px.box(
    barra_df_08['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN (08 Data)'
)
fig.show()

# Database Schema Design
- Create at least 3 tables:
  - 2007 Barra USE3 data
  - 2008 Barra USE3 data  
  - Industry-to-Sector mapping (from PDF pages 83-84)
- Use the Industry-to-Sector table for all sector-related queries
- Address column naming issues (columns with '%')
- Optimize data types with justification
- Implement primary keys and foreign key relationships
- Create both single and composite indices

In [ ]:
# GIVEN
SECTORS = {
    'FINANCIAL': ['BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS'],
    'TECH_COMM': ['SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS',
                  'ELECEQP', 'TELEPHON', 'WIRELESS', 'MEDIA', 'PUBLISH'],
    'REALESTATE_UTIL': ['EQTYREIT', 'ELECUTIL', 'GASUTIL'],
    'HEALTHCARE': ['MEDPRODS', 'BIOTECH', 'DRUGS', 'MEDPROVR'],
    'CONS_DISCR': ['SPLTYRET', 'RESTRNTS', 'CLOTHING', 'APPAREL', 'DEPTSTOR',
                   'HOTELS', 'ENTRTAIN', 'LEISURE', 'MOTORVEH', 'CONSDUR'],
    'CONS_STAPLES': ['GROCERY', 'FOODBEV', 'HOMEPROD', 'ALCOHOL', 'TOBACCO'],
    'INDUSTRIAL': ['CONSTRUC', 'INDPART', 'INDSVCS', 'HEAVYELC', 'HEAVYMCH',
                  'DEFAERO', 'ENVSVCS'],
    'BASIC_MAT': ['CHEMICAL', 'FOREST', 'GOLD'],
    'ENERGY': ['ENGYRES', 'OILSVCS', 'OILREF', 'MINING'],
    'TRANSPORT': ['AIRLINES', 'RAILROAD', 'TRUCKFRT']
}

In [ ]:
Image(image_folder / 'SECTOR_MAPPING_EXCERPT.jpg')

In [ ]:
# clean indname1 values for leading and trailing whitespace
barra_df_07['indname1'] = barra_df_07['indname1'].str.strip()
barra_df_08['indname1'] = barra_df_08['indname1'].str.strip()

In [ ]:
# get unique values for indname1 in both 07 and 08 to confirm sectors
unique_indname1_07 = barra_df_07['indname1'].unique()
unique_indname1_08 = barra_df_08['indname1'].unique()

In [ ]:
unique_indname1_07.sort()
unique_indname1_07

In [ ]:
# compare unique indname1 values to SECTORS dict values to confirm all sectors are represented
compare_df_to_sector_dict_07 = set(unique_indname1_07).difference({sector for sectors in SECTORS.values() for sector in sectors})
compare_df_to_sector_dict_08 = set(unique_indname1_08).difference({sector for sectors in SECTORS.values() for sector in sectors})

compare_sector_to_df_07 = set({sector for sectors in SECTORS.values() for sector in sectors}).difference(unique_indname1_07)
compare_sector_to_df_08 = set({sector for sectors in SECTORS.values() for sector in sectors}).difference(unique_indname1_08)

In [ ]:
display(f"Non matching: df -> sector dict 07: {compare_df_to_sector_dict_07}")
display(f"non matching: sector dict -> df 07: {compare_sector_to_df_07}")

In [ ]:
display(f"Non matching: df -> sector dict 08: {compare_df_to_sector_dict_08}")
display(f"non matching: sector dict -> df 08: {compare_sector_to_df_08}")

### ADDING SUFFIX to column names

In [ ]:
def add_suffix_to_cols(df, suffix):
    df.columns = [f"{col}_{suffix}" for col in df.columns]
    return df

In [ ]:
barra_df_07 = add_suffix_to_cols(barra_df_07, '07')
barra_df_08 = add_suffix_to_cols(barra_df_08, '08')

In [ ]:
# String Definitions, adding 50% buffer and rounding up with math.ceil
SQL_TYPE_MAPPING = {
    'int64': generic_sql_types.Integer,
    'float64': generic_sql_types.Float,
}

def generate_sqlalchemy_schema(meta_data, buffer = 1.5):
    """
    given a dictionary with col as keys and a nested dict with dtype and max length,
    we build a new dictionary that adds a 50% buffer to string lengths,
    if it is int64 or float64 we use a SQL_TYPE_MAPPING constants to use sqlalchemy types,
    we previously labeled columsn with values between 0 and 1 as boolean, so this handles it.  
    Buffer is added for future growth. 
    
    """
    
    orm_schema = dict()

    for col, info in meta_data.items():
        dtype = info['dtype']
        
        match dtype:
            case 'object':
                max_length = info['max_length']
                buffer_length = math.ceil(max_length * buffer) # add buffer% buffer and round up
                orm_schema[col] = generic_sql_types.String(buffer_length)
            case 'int64' | 'float64':
                orm_schema[col] = SQL_TYPE_MAPPING[dtype]
            case 'boolean':
                orm_schema[col] = generic_sql_types.Boolean
            case _:
                print(f"Data type {dtype} for column {col} not recognized.")
                
    return orm_schema


In [ ]:
def get_schema(df: pd.DataFrame) -> dict:
    """
    loop through a dataframe looking at each column and get its dtype name and length.
    """
    schema_dictonary = dict()
    for col in df.columns:
        dtype_name = df[col].dtype.name

        # create a nest dictionary for each column with dtype and length
        column_info = {"dtype": dtype_name}
        
        if dtype_name == 'object':
            # get the max length of strings in the column
            max_len = df[col].astype(str).str.len().max()
            column_info["max_length"] = int(max_len) if not pd.isna(max_len) else 0

        elif dtype_name in ['int64', 'float64']:
            unique_values = set(df[col].dropna().unique())
            if unique_values.issubset({0, 1}):
                column_info["dtype"] = 'boolean'

        schema_dictonary[col] = column_info
    return schema_dictonary
            

In [ ]:
type_dict_07 = get_schema(barra_df_07)
type_dict_08 = get_schema(barra_df_08)

In [ ]:
def get_string_max_lengths(type_dictionary):
    length_dict = dict()
    for col, info in type_dictionary.items():
        if info['dtype'] == 'object':
            length_dict[col] = info.get('max_length', 'N/A')
    return length_dict

In [ ]:
max_string_lengths_07 = get_string_max_lengths(type_dict_07)
max_string_lengths_08 = get_string_max_lengths(type_dict_08)
print(f"Max string lengths for 07 data:\n {max_string_lengths_07}")
print(f"Max string lengths for 08 data:\n {max_string_lengths_08}")


In [ ]:
# creating two dictionaries. note year has been added to suffix of columns earlier.
DTYPE_DICT_07 = generate_sqlalchemy_schema(type_dict_07) 
DTYPE_DICT_08 = generate_sqlalchemy_schema(type_dict_08) 

### Column Naming:
We decided to keep the column names the same, except where there were invalid characters.
The reason for this was that perhaps this columns are part of a standard data dictionary.
What we did decide to do was append the year to the columns so that we aren't reusing column names.
However, all new columns created by us would follow these rules:
- be given a name that suggests the purpose of the column and identify the characteristic of the stock to be represented
- Semantics:
    - the name should be meaningful and descriptive
    - clear and unambigous 
    - avoid acronyms
    - use abbreviations only if it enhances the column name
    - don't use a name that implicitly/explicitly identifies more than one characteristic.
    - use the singular form
    - not dynamic

### Column constraints:
We set the ticket to be the primary key for each table created.<br>
We set a constraint on CUSIP to be unique and not null.<br>

### Schema
The relationship we did establish was between the stocks and their risk factors to the sector mapping table.<br>
We chose the ticker to be the foreign key on the sector_mapping. <br>
We decided to add two columns, "created_on" and "updated_on".

### Additional Indexes
During the course of the project, we will review which columns are being most used to guide our decision in optimizing performance.<br>
For example, if we find ourselves ranking stocks by a risk factor, we will create an index on that column to speed up sorting.<br>
Another example would be if we find ourselves constantly looking for specific columns for a given stock, we will create a covered query -- a composite index, so that the database doesn't have to do a full scan.

In [ ]:
DTYPE_DICT_07

In [ ]:
def create_table_def_from_dtype_dict(dtype_dict):
    table_df = list()
    for col, dtype in dtype_dict.items():
        match col:
            case 'ticker':
                table_df.append(Column(col, dtype, primary_key=True))
            case 'cusip':
                table_df.append(Column(col, dtype, unique=True, nullable=False))
            case _:
                table_df.append(Column(col, dtype))
    return table_df

In [ ]:
table_07 = create_table_def_from_dtype_dict(DTYPE_DICT_07)
table_08 = create_table_def_from_dtype_dict(DTYPE_DICT_08)
print(f"{table_07[:2]=}")
print(f"{table_08[:2]=}")

**NOTE**: CUSIP doesn't show UNIQUE constraint because it's created in the background. We will check this after association with a metadata object.

**NOTE**: Because industry will be commonly searched on, we will create a covered query (a composite index) between Industry and sector. <br>
I.e: 'BANKS' --> 'FINANCIAL'

In [ ]:
# find max length of a string in a dictionary
max_length_string_in_sector_dict= 0 
for sector, industries in SECTORS.items():
    if len(sector) > max_length_string_in_sector_dict:
        max_length_string_in_sector_dict = len(sector)
    for industry in industries:
        if len(industry) > max_length_string_in_sector_dict:
            max_length_string_in_sector_dict = len(industry)
print(f"Max string length across all sectors and industries: {max_length_string_in_sector_dict}")

In [ ]:
flattened_industry_to_sector_mapping = list()
for sector, industries in SECTORS.items():
    for industry in industries:
        flattened_industry_to_sector_mapping.append(
            (industry, sector)
        )
flattened_industry_to_sector_mapping_df = pd.DataFrame(flattened_industry_to_sector_mapping, columns=['industry', 'sector'])
flattened_industry_to_sector_mapping_df.head(5)

In [ ]:
# review dfs to be inserted
display(barra_df_07.head(1))
display(barra_df_08.head(1))
display(flattened_industry_to_sector_mapping_df.head(1))

### Potential Problem
Database file may already have been created and populated with data. <br>
Two approaches:
1. check if record counts in passed df match record count in destination table. It provides some protection.
2. update bulk_insert_dataframe to either update or insert --upsert. 

In [ ]:
import select


def bulk_upsert_dataframe_into_sqlite(engine, table_object, df):
    from sqlalchemy.dialects.sqlite import insert as sqlite_upsert
    records = df.to_dict(orient='records')

    # get count before
    with engine.connect() as connection:
        count_before = connection.execute(sqlalchemy_select(sqlalchemy_func.count()).select_from(table_object))

    # create the insert statement
    sql_stmt = sqlite_upsert(table_object).values(records)

    # do nothing if primary key conflict occurs
    upsert_stmt = sql_stmt.on_conflict_do_nothing()

    # execute TRANSACTION
    with engine.begin() as connection:
        result = connection.execute(upsert_stmt)
        rows_affected = result.rowcount

    # get count after
    with engine.connect() as connection:
        count_after = connection.execute(
            sqlalchemy_select(sqlalchemy_func.count()).select_from(table_object)
            )

    expected_total = count_before + rows_affected
    assert count_after == expected_total, (
        f"{count_after=} does not match {expected_total=}"
    )

    if count_before == 0:
        assert rows_affected == len(df), f"load failure: expected {len(df)=}, but got {rows_affected=}"

    logger.info(f"UPSERT complete: {rows_affected=} | Total: {count_after}")

### NOTE ON CONNECTION ENGINE
SQLITE:
- 3 slashes is relative paths
- 4 slashes is for absolute paths <br>
reference: https://docs.sqlalchemy.org/en/21/core/engines.html#sqlite

In [ ]:
# create a SQLit3 engine
db_file_path = curr_working_dir / 'barra_data.db'
engine = create_engine(f'sqlite:///{db_file_path}')

In [ ]:
metadata = MetaData()

table_barra_07 = Table(
    'barra_07',
    metadata,
    *table_07,
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now)
)

table_barra_08 = Table(
    'barra_08',
    metadata,
    *table_08,
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now)
)

buffer_length_for_industry_sector = math.ceil(max_length_string_in_sector_dict * 1.5)
table_industry_sector_mapping = Table(
    'industry_to_sector',
    metadata,
    Column('industry', generic_sql_types.String(buffer_length_for_industry_sector), primary_key=True),
    Column('sector', generic_sql_types.String(buffer_length_for_industry_sector), nullable=False),
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now),

    # create a covered index for performance
    Index('idx_industry_sector', 'industry', 'sector')
)

metadata.create_all(engine)

In [ ]:
# load 07 and 08 data into their respective tables and load industry to sector mapping into its table
bulk_upsert_dataframe_into_sqlite(engine, table_barra_07, barra_df_07)
#bulk_upsert_dataframe_into_sqlite(engine, table_barra_08, barra_df_08)
#bulk_upsert_dataframe_into_sqlite(engine, table_industry_sector_mapping, flattened_industry_to_sector_mapping_df)

In [ ]:
### Verify data was insert correctly using SQL queries with sqlalchemy core using sqlalchemy TEXT object
count_query_07_query = 